# Fall alerts in a care home: training pipeline

A camera image of a resident's room is checked for someone who has fallen. When the system
raises an alert, the night nurse goes to the room. **Falling is the positive class** throughout.

This notebook downloads the dataset, describes it, finetunes `yolo11n`, and evaluates it per
class on the test split. Baselines, the cost sweep and the human-review band follow below.

Run it end to end with `uv run python main.py train`, or on Colab with a T4 GPU.

## 0. Setup

On Colab: Runtime > Change runtime type > T4, and add a secret named `ROBOFLOW_API_KEY` (key
icon in the sidebar) with notebook access switched on. Locally the key comes from `.env`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab starts in an empty folder, so fetch the repo to get api_key.py and the weights/ folder.
    if not Path("api_key.py").exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Allan-Data-science/daiist-assignment-2.git"],
            check=True,
        )
        os.chdir("daiist-assignment-2")
    # Plain pip, not %pip: main.py runs this notebook as an ordinary script, where magics do not
    # exist. No uv sync either, because Colab brings its own GPU build of PyTorch. Ultralytics is
    # pinned to the same range as pyproject.toml so the weights load in the local app.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.4,<8.5", "roboflow"],
        check=True,
    )
    # api_key.py checks the environment variable first, so the Colab secret goes there.
    from google.colab import userdata

    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

from api_key import load_api_key

API_KEY = load_api_key()
# Print the shape of the key, never the key.
print(f"key loaded: {len(API_KEY)} characters")

## 1. The dataset

| | |
|---|---|
| workspace | `falldetection-bwfdz` |
| project | `fall-detection-ehgga` |
| version | `1` |
| classes | `Falling`, `Sitting`, `Standing` |
| licence | CC BY 4.0 |

The version number is pinned so the same images come back every time.

A second public dataset is used **only as an external test set**, never for training. Section
1.2 shows why: the dataset's own test split is made of near-copies of training images.

| | |
|---|---|
| workspace | `ynt423` |
| project | `labels-6tnj1` |
| version | `1` (test split only) |
| classes | `Fall`, `Sitting`, `Standing`, in the same order as ours |
| licence | CC BY 4.0 |

In [ ]:
import yaml
from roboflow import Roboflow


def download(workspace, project, version):
    dataset_dir = Path(f"datasets/{project}-{version}").resolve()
    if not dataset_dir.exists():
        rf = Roboflow(api_key=API_KEY)
        rf.workspace(workspace).project(project).version(version).download(
            "yolov11", location=str(dataset_dir)
        )
    # Roboflow writes split paths like "../train/images", which point outside the dataset
    # folder. Rewrite them relative to an explicit root so Ultralytics finds the images.
    data_yaml = dataset_dir / "data.yaml"
    config = yaml.safe_load(data_yaml.read_text())
    config.update(path=str(dataset_dir), train="train/images", val="valid/images",
                  test="test/images")
    data_yaml.write_text(yaml.safe_dump(config))
    return dataset_dir, data_yaml, config["names"]


DATASET_DIR, DATA_YAML, CLASS_NAMES = download("falldetection-bwfdz", "fall-detection-ehgga", 1)
EXTERNAL_DIR, EXTERNAL_YAML, EXTERNAL_NAMES = download("ynt423", "labels-6tnj1", 1)

POSITIVE = "Falling"
# Both datasets index Fall(ing), Sitting, Standing as 0, 1, 2, so our model's class ids apply to
# the external labels unchanged. Stop here if that ever stops being true.
assert [n[:4].lower() for n in EXTERNAL_NAMES] == [n[:4].lower() for n in CLASS_NAMES]
print(CLASS_NAMES, EXTERNAL_NAMES)

### 1.1 Describe it before training on it

Instances per class, images per split, and how much of the image a person covers. Roboflow
augmented the training split, so it also counts how many *original* photographs are behind it:
augmented copies share the part of the file name before `.rf.`.

In [ ]:
from collections import Counter


def describe_split(dataset_dir, split):
    label_files = sorted((dataset_dir / split / "labels").glob("*.txt"))
    instances, box_areas, empty, crowded = Counter(), [], 0, 0
    for f in label_files:
        # A YOLO label file is one box per line: "class_id x y w h", all relative to the image.
        lines = [line.split() for line in f.read_text().splitlines() if line.strip()]
        if not lines:
            empty += 1
        if len(lines) > 1:
            crowded += 1
        for parts in lines:
            instances[CLASS_NAMES[int(parts[0])]] += 1
            coords = [float(x) for x in parts[1:]]
            if len(coords) == 4:
                w, h = coords[2], coords[3]
            else:
                # Some external labels are polygons ("cls x1 y1 x2 y2 ..."): use their bounding box.
                w, h = max(coords[0::2]) - min(coords[0::2]), max(coords[1::2]) - min(coords[1::2])
            box_areas.append(w * h)
    originals = {f.name.split(".rf.")[0] for f in label_files}
    box_areas.sort()
    return {
        "images": len(label_files),
        "originals": len(originals),
        "empty": empty,
        "multi_person": crowded,
        **{name: instances[name] for name in CLASS_NAMES},
        "median_box": box_areas[len(box_areas) // 2],
    }


dataset_stats = {split: describe_split(DATASET_DIR, split) for split in ("train", "valid", "test")}
dataset_stats["external"] = describe_split(EXTERNAL_DIR, "test")

print(f"{'split':<10}{'images':>8}{'originals':>11}{'empty':>7}{'2+ people':>11}", end="")
print("".join(f"{name:>10}" for name in CLASS_NAMES) + f"{'median box':>12}")
for split, s in dataset_stats.items():
    print(f"{split:<10}{s['images']:>8}{s['originals']:>11}{s['empty']:>7}{s['multi_person']:>11}",
          end="")
    print("".join(f"{s[name]:>10}" for name in CLASS_NAMES) + f"{s['median_box']:>12.0%}")

### 1.2 Are the test images really new?

File names cannot answer this: burst photos taken a second apart get different numbers. Instead
each image is reduced to a 256-bit *difference hash* (shrink to 17x16 greyscale, record whether
each pixel is brighter than its right-hand neighbour), and every test image is compared with its
nearest training image. A distance of a few bits out of 256 means practically the same picture.

In [ ]:
import numpy as np
from PIL import Image


def image_hash(path):
    grey = np.asarray(Image.open(path).convert("L").resize((17, 16)), dtype=np.int16)
    return (grey[:, 1:] > grey[:, :-1]).flatten()


train_hashes = np.array([image_hash(p) for p in (DATASET_DIR / "train" / "images").glob("*")])


def nearest_train_distance(image_dir):
    return np.array([(train_hashes != image_hash(p)).sum(1).min() for p in image_dir.glob("*")])


near_duplicates = {}
print(f"{'split':<10}{'images':>8}{'within 10 bits':>16}{'median distance':>17}")
for split, image_dir in [("valid", DATASET_DIR / "valid" / "images"),
                         ("test", DATASET_DIR / "test" / "images"),
                         ("external", EXTERNAL_DIR / "test" / "images")]:
    d = nearest_train_distance(image_dir)
    near_duplicates[split] = {"images": len(d), "within_10": int((d <= 10).sum()),
                              "median": int(np.median(d))}
    print(f"{split:<10}{len(d):>8}{int((d <= 10).sum()):>16}{int(np.median(d)):>17}")

## 2. Finetune yolo11n

Start from the COCO-pretrained checkpoint, so the model already knows what a person looks like
and only has to learn the posture. `patience` stops training once validation mAP has not improved
for that many epochs, which guards against overfitting a training set of only 656 originals.

`EPOCHS` can be overridden from the environment for a quick smoke test (`EPOCHS=1`).

In [ ]:
import shutil

from ultralytics import YOLO

EPOCHS = int(os.environ.get("EPOCHS", 50))
IMGSZ, BATCH, PATIENCE, SEED = 640, 16, 10, 0

model = YOLO("yolo11n.pt")
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    project=str(Path("runs").resolve()),
    name="fall",
    exist_ok=True,
)

# runs/ is not committed. The app and the grader load weights/best.pt, so copy it there.
best_run_weights = Path(model.trainer.save_dir) / "weights" / "best.pt"
WEIGHTS = Path("weights/best.pt")
shutil.copy(best_run_weights, WEIGHTS)
print(f"saved {WEIGHTS} ({WEIGHTS.stat().st_size / 1e6:.1f} MB)")

## 3. Per-class results on both test sets

Neither test set was used for training or for choosing when to stop. The internal test split is
reported because the brief asks for it, but Section 1.2 showed it is made of near-copies of
training images, so it is an upper bound. The external test set is the realistic number. The
overall mAP is printed, but the row that matters is `Falling`.

In [ ]:
import json

finetuned = YOLO(str(WEIGHTS))


def class_metrics(data_yaml, name):
    # Box-level precision, recall and AP per class, plus the confusion matrix, on one test split.
    result = finetuned.val(data=str(data_yaml), split="test", imgsz=IMGSZ, plots=True, name=name)
    box = result.box
    per_class = {
        cls: {"precision": float(box.p[i]), "recall": float(box.r[i]),
              "ap50": float(box.ap50[i]), "ap50_95": float(box.maps[i])}
        for i, cls in enumerate(CLASS_NAMES)
    }
    print(f"{name}: overall  mAP50 {box.map50:.3f}   mAP50-95 {box.map:.3f}")
    print(f"{'class':<10}{'precision':>11}{'recall':>9}{'AP50':>8}{'AP50-95':>10}")
    for cls, m in per_class.items():
        print(f"{cls:<10}{m['precision']:>11.3f}{m['recall']:>9.3f}"
              f"{m['ap50']:>8.3f}{m['ap50_95']:>10.3f}")
    print(f"confusion matrix plot: {result.save_dir}/confusion_matrix.png\n")
    return {
        "overall": {"map50": float(box.map50), "map50_95": float(box.map)},
        "per_class": per_class,
        # Rows are predicted class, columns are true class; the last row/column is background.
        "confusion_matrix": result.confusion_matrix.matrix.astype(int).tolist(),
    }


test_results = {"internal": class_metrics(DATA_YAML, "internal"),
                "external": class_metrics(EXTERNAL_YAML, "external")}

# Everything the app needs later goes in outputs/, which is committed alongside the weights.
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)
(OUTPUTS / "test_metrics.json").write_text(
    json.dumps(
        {
            "dataset": dataset_stats,
            "near_duplicates": near_duplicates,
            **test_results,
            "matrix_labels": CLASS_NAMES + ["background"],
            "train_settings": {"epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
                               "patience": PATIENCE, "seed": SEED},
        },
        indent=2,
    )
)

## 4. From boxes to one decision per image

The nurse does not act on boxes, she acts on one question per image: *has someone fallen?* So
**one image is one case**, and an image counts as a fall if anyone in it has fallen. (Every
internal image shows one person; a third of the external images show several.) Every system
below is reduced to a single **fall score** per image, and an alert is raised when the score is
at or above a cut:

| system | fall score |
|---|---|
| finetuned yolo11n | confidence of the most confident `Falling` box, 0 if none |
| A. COCO, any person | confidence of the most confident `person` box, 0 if none |
| C. COCO, wide box (bonus) | most confident `person` box that is wider than tall, 0 if none |
| predict nothing | always 0, so never an alert |

Taking the maximum, not the mean: one confident fall box is enough evidence to send the nurse.

Predictions are kept down to 1% confidence, so the borderline cases are not thrown away before the
threshold sweep sees them.

In [ ]:
coco = YOLO("yolo11n.pt")  # the same checkpoint we finetuned from, used as it is
COCO_PERSON = 0  # "person" is class 0 in COCO
FALLING = CLASS_NAMES.index(POSITIVE)
SYSTEMS = ["finetuned", "coco_person", "coco_wide", "nothing"]


def max_conf(boxes):
    return max((float(b.conf) for b in boxes), default=0.0)


def is_fall(label_path):
    # Ground truth for one image: does any label in it say Fall(ing)?
    ids = {int(line.split()[0]) for line in label_path.read_text().splitlines() if line.strip()}
    return FALLING in ids


def score_test_set(dataset_dir):
    cases = []
    for path in sorted((dataset_dir / "test" / "images").glob("*")):
        case = {"image": path.name,
                "truth": is_fall(dataset_dir / "test" / "labels" / f"{path.stem}.txt")}

        ours = finetuned.predict(path, conf=0.01, imgsz=IMGSZ, verbose=False)[0].boxes
        case["finetuned"] = max_conf([b for b in ours if int(b.cls) == FALLING])

        people = [b for b in coco.predict(path, conf=0.01, verbose=False)[0].boxes
                  if int(b.cls) == COCO_PERSON]
        case["coco_person"] = max_conf(people)
        # xywh is centre x, centre y, width, height. Someone lying on the floor is wider than tall.
        case["coco_wide"] = max_conf([b for b in people if b.xywh[0][2] > b.xywh[0][3]])

        case["nothing"] = 0.0
        cases.append(case)
    print(f"{dataset_dir.name}: {len(cases)} images, {sum(c['truth'] for c in cases)} with a fall")
    return cases


test_sets = {"internal": score_test_set(DATASET_DIR), "external": score_test_set(EXTERNAL_DIR)}

### 4.1 The four systems on both test sets

Falling is positive. The tables are at two example cuts only; the cut actually used is chosen by
cost in the next section, from the same scores.

In [ ]:
def confusion(cases, system, cut):
    # Positive class is a fall. A score at or above the cut raises an alert.
    tp = sum(1 for c in cases if c[system] >= cut and c["truth"])
    fp = sum(1 for c in cases if c[system] >= cut and not c["truth"])
    fn = sum(1 for c in cases if c[system] < cut and c["truth"])
    tn = sum(1 for c in cases if c[system] < cut and not c["truth"])
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "recall": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "precision": tp / (tp + fp) if tp + fp else float("nan"),
        "accuracy": (tp + tn) / len(cases),
    }


for name, cases in test_sets.items():
    for cut in (0.25, 0.5):
        print(f"{name} test set, cut {cut}")
        print(f"{'system':<13}{'TP':>5}{'FP':>5}{'FN':>5}{'TN':>5}"
              f"{'recall':>9}{'specificity':>13}{'precision':>11}{'accuracy':>10}")
        for system in SYSTEMS:
            s = confusion(cases, system, cut)
            precision = "-" if s["precision"] != s["precision"] else f"{s['precision']:.1%}"
            print(f"{system:<13}{s['tp']:>5}{s['fp']:>5}{s['fn']:>5}{s['tn']:>5}"
                  f"{s['recall']:>9.1%}{s['specificity']:>13.1%}{precision:>11}"
                  f"{s['accuracy']:>10.1%}")
        print()

# Per-image scores are what the cost sweep and the dashboard are built from.
(OUTPUTS / "test_scores.json").write_text(json.dumps(test_sets, indent=2))

## 5. Cost per 1,000 cases and the threshold sweep

The cost of each outcome, argued in REPORT.md:

| outcome | cost | what happens |
|---|---|---|
| **TP** | EUR 5 | nurse goes to the room and finds the fall |
| **FP** | EUR 5 | nurse goes to the room and finds no fall |
| **FN** | EUR 2,000 | fall missed, resident lies on the floor until the next round |
| **TN** | EUR 0 | no fall, no alert |

**Base rate: 1%** of the images the system checks contain a fall. Neither test set has that mix
(internal 33%, external 18%), so their precision and accuracy do not transfer. Recall and
specificity are properties of the model and do, so they are measured on each test set and
projected onto the real mix:

```
TP = p * recall                   FN = p * (1 - recall)
FP = (1 - p) * (1 - specificity)  TN = (1 - p) * specificity
```

In [ ]:
COST_TP, COST_FP, COST_FN, COST_TN = 5.0, 5.0, 2000.0, 0.0
BASE_RATE = 0.01

# Cuts from 0.01 to 1.01 (never alert) in steps of 0.01. A cut of 0 is left out: a score of 0
# means the model found no box at all, and alerting on that is "check every image", which is
# priced separately as the everyone-reviews design, not as a model threshold.
CUTS = [round(i / 100, 2) for i in range(1, 102)]


def cost_per_1000(stats, base_rate=BASE_RATE, cost_fn=COST_FN):
    falls, others = 1000 * base_rate, 1000 * (1 - base_rate)
    tp, fn = falls * stats["recall"], falls * (1 - stats["recall"])
    fp, tn = others * (1 - stats["specificity"]), others * stats["specificity"]
    return tp * COST_TP + fp * COST_FP + fn * cost_fn + tn * COST_TN


def sweep(cases, system, base_rate=BASE_RATE, cost_fn=COST_FN):
    # One row per cut: the cut, its expected cost per 1,000 cases, and the test confusion.
    rows = []
    for cut in CUTS:
        s = confusion(cases, system, cut)
        rows.append({"cut": cut, "cost": cost_per_1000(s, base_rate, cost_fn), **s})
    return rows


def cheapest(rows):
    return min(rows, key=lambda r: r["cost"])


sweeps = {name: {system: sweep(cases, system) for system in SYSTEMS}
          for name, cases in test_sets.items()}

for name in test_sets:
    print(f"{name} test set: each system at its own cheapest cut, base rate {BASE_RATE:.0%}")
    print(f"{'system':<13}{'cut':>6}{'recall':>9}{'specificity':>13}{'EUR/1,000':>12}")
    for system in SYSTEMS:
        best = cheapest(sweeps[name][system])
        print(f"{system:<13}{best['cut']:>6}{best['recall']:>9.1%}"
              f"{best['specificity']:>13.1%}{best['cost']:>12,.2f}")
    print()

In [ ]:
import matplotlib

matplotlib.use("Agg")  # no window: main.py runs this as a plain script
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, name in zip(axes, test_sets):
    for system in SYSTEMS:
        rows = sweeps[name][system]
        ax.plot([r["cut"] for r in rows], [r["cost"] for r in rows], label=system)
    best = cheapest(sweeps[name]["finetuned"])
    ax.scatter([best["cut"]], [best["cost"]], color="black", zorder=3,
               label=f"cheapest finetuned cut {best['cut']}")
    ax.set_title(f"{name} test set")
    ax.set_xlabel("confidence threshold (alert at or above)")
    ax.legend()
axes[0].set_ylabel("expected cost, EUR per 1,000 cases")
fig.suptitle(f"Cost against threshold at a {BASE_RATE:.0%} base rate")
fig.tight_layout()
fig.savefig(OUTPUTS / "cost_curve.png", dpi=120)

### 5.1 Sensitivity: how much does the answer depend on the two weakest assumptions?

The FN cost and the base rate are the least certain figures. Each is varied with everything else
held fixed, and the cheapest cut for the finetuned model is found again. The cheapest cut moving
is expected. What matters is the point where the decision itself changes: the system is no longer
worth running at all, or a baseline becomes cheaper.

In [ ]:
def best_by_system(cases, base_rate=BASE_RATE, cost_fn=COST_FN):
    return {s: cheapest(sweep(cases, s, base_rate, cost_fn)) for s in SYSTEMS}


def print_sensitivity(cases, label, values, fmt):
    print(f"{label:>12}{'best cut':>10}{'recall':>9}{'EUR/1,000':>12}"
          f"{'coco_wide':>12}{'nothing':>11}{'cheapest system':>17}")
    rows = []
    for value in values:
        best = best_by_system(cases, **value)
        ours = best["finetuned"]
        winner = min(best, key=lambda s: best[s]["cost"])
        print(f"{fmt(value):>12}{ours['cut']:>10}{ours['recall']:>9.1%}{ours['cost']:>12,.2f}"
              f"{best['coco_wide']['cost']:>12,.2f}{best['nothing']['cost']:>11,.2f}{winner:>17}")
        rows.append({**value, **{s: best[s]["cost"] for s in SYSTEMS},
                     "best_cut": ours["cut"], "winner": winner})
    print()
    return rows


FN_COSTS = [250, 500, 1000, 2000, 5000, 10000]
BASE_RATES = [0.001, 0.0025, 0.005, 0.01, 0.02, 0.05]

sensitivity = {}
for name, cases in test_sets.items():
    print(f"{name} test set: FN cost varied, base rate fixed at {BASE_RATE:.0%}")
    fn_rows = print_sensitivity(cases, "FN cost", [{"cost_fn": c} for c in FN_COSTS],
                                lambda v: f"EUR {v['cost_fn']:,}")
    print(f"{name} test set: base rate varied, FN cost fixed at EUR {COST_FN:,.0f}")
    rate_rows = print_sensitivity(cases, "base rate", [{"base_rate": r} for r in BASE_RATES],
                                  lambda v: f"{v['base_rate']:.2%}")
    sensitivity[name] = {"fn_cost": fn_rows, "base_rate": rate_rows}

In [ ]:
(OUTPUTS / "cost_sweep.json").write_text(
    json.dumps(
        {
            "costs": {"tp": COST_TP, "fp": COST_FP, "fn": COST_FN, "tn": COST_TN},
            "base_rate": BASE_RATE,
            **{name: {"sweeps": sweeps[name],
                      "cheapest": {s: cheapest(sweeps[name][s]) for s in SYSTEMS},
                      "sensitivity": sensitivity[name]}
               for name in test_sets},
        },
        indent=2,
    )
)

## 6. Pass, human review, fail

To do: the two thresholds, review load and staffing.